# CV Generator — Backend

1. Rode a célula abaixo para instalar dependências e iniciar o servidor.
2. Copie a URL do ngrok que aparece na saída.
3. No site [CV Generator](https://xangrybadger.github.io/cv-generator/), clique em **Sem API** no header e cole a URL.

A URL muda a cada sessão — você precisa colar novamente se reiniciar o notebook.

In [ ]:
!pip install -q fastapi uvicorn reportlab python-multipart pydantic pyngrok

In [ ]:
import os
os.makedirs('app', exist_ok=True)

%%writefile app/main.py
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.lib.colors import HexColor
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable
from reportlab.lib.enums import TA_LEFT, TA_JUSTIFY
import io

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class CVData(BaseModel):
    name: str
    email: str
    phone: str
    location: str
    linkedin: str
    github: str
    portfolio: str
    title: str
    subtitle: str
    profile: str
    experience: str
    projects: str
    education: str
    skills: str
    language: str
    template: str

SAGE = HexColor("#456A4B")
AMBER = HexColor("#8B6914")
DARK = HexColor("#0D1117")
MUTED = HexColor("#8B9481")
BORDER = HexColor("#D4D0C8")

def make_styles():
    styles = {}
    styles["name"] = ParagraphStyle(
        "Name", fontName="Helvetica-Bold", fontSize=15,
        leading=18, textColor=DARK, alignment=TA_LEFT,
        spaceAfter=1*mm,
    )
    styles["title"] = ParagraphStyle(
        "Title", fontName="Helvetica", fontSize=8.5,
        leading=11, textColor=AMBER, alignment=TA_LEFT,
        spaceAfter=0.5*mm,
    )
    styles["contact"] = ParagraphStyle(
        "Contact", fontName="Helvetica", fontSize=7,
        leading=9.5, textColor=MUTED, alignment=TA_LEFT,
        spaceAfter=0,
    )
    styles["section_head"] = ParagraphStyle(
        "SectionHead", fontName="Helvetica-Bold", fontSize=9.5,
        leading=11.5, textColor=DARK, alignment=TA_LEFT,
        spaceBefore=2.5*mm, spaceAfter=1*mm,
    )
    styles["body"] = ParagraphStyle(
        "Body", fontName="Helvetica", fontSize=8,
        leading=11, textColor=DARK, alignment=TA_JUSTIFY,
        spaceAfter=0.3*mm,
    )
    styles["body_small"] = ParagraphStyle(
        "BodySmall", fontName="Helvetica", fontSize=7,
        leading=9.5, textColor=MUTED, alignment=TA_LEFT,
        spaceAfter=0.2*mm,
    )
    styles["bullet"] = ParagraphStyle(
        "Bullet", fontName="Helvetica", fontSize=8,
        leading=11, textColor=DARK, alignment=TA_JUSTIFY,
        leftIndent=8, bulletIndent=0,
        spaceAfter=0.4*mm,
    )
    return styles

def generate_pdf(data: CVData):
    buffer = io.BytesIO()
    styles = make_styles()
    story = []

    def s(key):
        return styles[key]

    def heading(text):
        story.append(Paragraph(text.upper(), s("section_head")))
        story.append(HRFlowable(
            width="100%", thickness=0.5, color=BORDER,
            spaceBefore=0, spaceAfter=2*mm,
        ))

    def bullet(text):
        story.append(Paragraph(f"<bullet>&bull;</bullet> {text}", s("bullet")))

    story.append(Paragraph(data.name, s("name")))
    story.append(Paragraph(f'{data.title} · {data.subtitle}', s("title")))

    contact_parts = [
        data.email, data.phone, data.location,
        data.linkedin, data.github, data.portfolio,
    ]
    story.append(Paragraph(" · ".join(contact_parts), s("contact")))
    story.append(Spacer(1, 1.5*mm))

    heading("Perfil" if data.language == 'pt' else "Profile")
    story.append(Paragraph(data.profile, s("body")))

    heading("Experiência" if data.language == 'pt' else "Experience")
    for line in data.experience.split('\n'):
        if line.strip():
            if line.startswith('•'):
                bullet(line[1:].strip())
            else:
                story.append(Paragraph(line, s("body")))
    story.append(Spacer(1, 1*mm))

    heading("Projetos" if data.language == 'pt' else "Projects")
    for line in data.projects.split('\n'):
        if line.strip():
            if line.startswith('•'):
                bullet(line[1:].strip())
            else:
                story.append(Paragraph(line, s("body")))
    story.append(Spacer(1, 1*mm))

    heading("Educação" if data.language == 'pt' else "Education")
    for line in data.education.split('\n'):
        if line.strip():
            story.append(Paragraph(line, s("body")))

    heading("Skills")
    story.append(Paragraph(data.skills, s("body")))

    doc = SimpleDocTemplate(
        buffer, pagesize=A4,
        topMargin=14*mm, bottomMargin=12*mm,
        leftMargin=16*mm, rightMargin=16*mm,
    )
    doc.build(story)
    buffer.seek(0)
    return buffer.getvalue()

@app.post("/api/generate-cv")
async def generate_cv(cv_data: CVData):
    try:
        pdf_bytes = generate_pdf(cv_data)
        return io.BytesIO(pdf_bytes)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/")
async def root():
    return {"message": "CV Generator API"}

In [ ]:
# Coloque seu token do ngrok aqui (grátis em https://ngrok.com/signup)
NGROK_TOKEN = ''  # ← Cole entre as aspas

from pyngrok import ngrok
import threading
import uvicorn
import sys

if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

def run():
    uvicorn.run('app.main:app', host='0.0.0.0', port=8000)

thread = threading.Thread(target=run)
thread.daemon = True
thread.start()

import time
time.sleep(2)

tunnel = ngrok.connect(8000)
public_url = tunnel.public_url
print(f'\n{"="*60}')
print(f'  CV Generator Backend Online!')
print(f'  URL: {public_url}')
print(f'{"="*60}')
print(f'\n  Cole esta URL no header do site:')
print(f'  https://xangrybadger.github.io/cv-generator/')
print()